# Data cleaning

In [57]:
from pathlib import Path
import pandas as pd

In [58]:
data_path = Path("..") / "data/raw/training_pop30_genres.parquet"
df = pd.read_parquet(data_path)
print(f"Loaded {len(df):,} tracks from {data_path}")

Loaded 1,863,421 tracks from ../data/raw/training_pop30_genres.parquet


In [59]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1863421 entries, 0 to 1863420
Data columns (total 34 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   track_rowid             int64  
 1   track_name              object 
 2   artist_name             object 
 3   artist_rowid            int64  
 4   album_rowid             int64  
 5   album_name              object 
 6   album_type              object 
 7   label                   object 
 8   release_date            object 
 9   release_date_precision  object 
 10  id_isrc                 object 
 11  id_upc                  object 
 12  time_signature          int64  
 13  tempo                   float64
 14  key                     int64  
 15  mode                    int64  
 16  danceability            float64
 17  energy                  float64
 18  loudness                float64
 19  speechiness             float64
 20  acousticness            float64
 21  instrumentalness        float64

## Fixing NaNs

In [4]:
100 * (missing:=df.isna().sum())[missing > 0] / len(df), missing[missing > 0]

(id_isrc           0.007620
 id_upc            0.274549
 artist_genres    34.772228
 dtype: float64,
 id_isrc             142
 id_upc             5116
 artist_genres    647953
 dtype: int64)

Drops `id_upc` column, and rows with NaN `id_isrc`.

In [5]:
df = df.drop("id_upc", axis="columns")
df = df.dropna(subset=("id_isrc",)).reset_index(drop=True)

In [6]:
100 * (missing:=df.isna().sum())[missing > 0] / len(df), missing[missing > 0]

(artist_genres    34.774127
 dtype: float64,
 artist_genres    647939
 dtype: int64)

Fills missing genres with standard token.

In [7]:
df["artist_genres"] = df["artist_genres"].fillna(value="<GENRE_UNKNOWN>")

In [10]:
df.isna().sum().sum()

np.int64(0)

## Duplicates

We have [ISRC](https://en.wikipedia.org/wiki/International_Standard_Recording_Code) for each track. The first thing we should probably do is to drop duplicates from this column.

In [86]:
_df = df[df["artist_rowid"] == 5587713] # Queen, large historical catalog
_df[["track_name", "album_name", "id_isrc", "release_date"]][_df["track_name"].str.lower().str.startswith("we will rock")]

,track_name,album_name,id_isrc,release_date
8784,We Will Rock You - Remastered 2011,Pra comemorar os esportes,GBUM71029618,2024-08-06
99606,We Will Rock You VonLichten,Ultimate Football Music Vol 1,QZ6ZQ1860690,2022-11-11
114057,We Will Rock You - Remastered,Greatest Hits - We Will Rock You Edition,GBCEE0100127,2010-06-15
274388,We Will Rock You VonLichten,We Will Rock You VonLichten,USHR11234040,2012-01-01
286675,We Will Rock You (Fast) - BBC Session / Octobe...,Queen On Air,GBCAD1600789,2016-10-21
324763,We Will Rock You - Remastered 2011,Queen Jewels,GBUM71029618,2004-01-01
381370,We Will Rock You - Remastered 2011,The Platinum Collection (2011 Remaster),GBUM71029618,2011-01-01
468370,We Will Rock You - Raw Sessions Version,We Will Rock You (Raw Sessions Version),GBUM71703576,2017-10-06
589903,We Will Rock You (Fast) - BBC Session / Octobe...,On Air,GBCAD1600789,2016-11-04
731176,We Will Rock You (Fast) - Live,Queen Rock Montreal,GBCEE0700003,2007-10-09


In [87]:
duplicated_id_isrc = _df[_df.duplicated("id_isrc", keep=False)][["track_name", "album_name", "id_isrc", "release_date"]]
duplicated_id_isrc[duplicated_id_isrc["track_name"].str.lower().str.startswith("we will")]

,track_name,album_name,id_isrc,release_date
8784,We Will Rock You - Remastered 2011,Pra comemorar os esportes,GBUM71029618,2024-08-06
114057,We Will Rock You - Remastered,Greatest Hits - We Will Rock You Edition,GBCEE0100127,2010-06-15
286675,We Will Rock You (Fast) - BBC Session / Octobe...,Queen On Air,GBCAD1600789,2016-10-21
324763,We Will Rock You - Remastered 2011,Queen Jewels,GBUM71029618,2004-01-01
381370,We Will Rock You - Remastered 2011,The Platinum Collection (2011 Remaster),GBUM71029618,2011-01-01
589903,We Will Rock You (Fast) - BBC Session / Octobe...,On Air,GBCAD1600789,2016-11-04
1135961,We Will Rock You - Remastered 2011,Queen 40 Limited Edition Collector's Box Set V...,GBUM71029618,2011-01-01
1136824,We Will Rock You - Remastered 2011,News Of The World (Deluxe Edition 2011 Remaster),GBUM71029618,1977-10-28
1226691,We Will Rock You - Megan Thee Stallion Version,We Will Rock You (Megan Thee Stallion Version),GBUM72403911,2024-09-05
1293083,We Will Rock You - Megan Thee Stallion Version,We Will Rock You (Megan Thee Stallion Version),GBUM72403911,2024-09-05


Looking at the dataset, it seems that spotify returns the last entries as first search matches. This could also be due to the fact that tracks are sorted by their popularity?
Anyway.. It seems we can just keep the last.

In [77]:
famous_reissued_songs = [
    "Stairway to Heaven",
    "Smells Like Teen Spirit",
    "Bohemian Rhapsody",
    "Money",
    "Comfortably Numb",
    "Yesterday",
    "A Day in the Life",
    "Gimme Shelter",
    "Like a Rolling Stone",
    "London Calling",
    "Purple Haze",
    "Heroes",
    "Superstition",
    "Dreams",
    "Born to Run",
    "Blue Monday",
    "Hotel California",
    "Under Pressure",
    "Losing My Religion",
    "Creep",
]

for title in famous_reissued_songs:
    if not df[df["track_name"].str.lower().str.startswith(title)]["track_popularity"].is_monotonic_increasing:
        print(f"{title} popularity is not monotonically increasing!!")

So let's drop the tracks with duplicated ISRC and duplicated artist/exact track name.

In [154]:
print(f"dataframe length: {len(df)}")
_df = df.drop_duplicates("id_isrc", keep="last").reset_index()
print(f"dataframe length after deduplicating id_isrc: {len(df)}")
_df = _df.drop_duplicates(["artist_rowid", "track_name"], keep="last").reset_index(drop=True)
print(f"dataframe length after deduplicating arist/track name combos: {len(df)}")

dataframe length: 1863421
dataframe length after deduplicating id_isrc: 1863421
dataframe length after deduplicating arist/track name combos: 1863421


In [113]:
_df[["track_name", "album_name", "id_isrc", "release_date"]][
    (_df["artist_name"] == "Queen") & 
    _df["track_name"].str.lower().str.startswith("we will rock")
]

,track_name,album_name,id_isrc,release_date
244631,We Will Rock You VonLichten,We Will Rock You VonLichten,USHR11234040,2012-01-01
417212,We Will Rock You - Raw Sessions Version,We Will Rock You (Raw Sessions Version),GBUM71703576,2017-10-06
525522,We Will Rock You (Fast) - BBC Session / Octobe...,On Air,GBCAD1600789,2016-11-04
651413,We Will Rock You (Fast) - Live,Queen Rock Montreal,GBCEE0700003,2007-10-09
1018883,"We Will Rock You - Live at Wembley Stadium, 13...","Queen at Live Aid (Live at Wembley Stadium, 13...",UK6821836639,2018
1146622,We Will Rock You - Live At Wembley Stadium / J...,Live At Wembley Stadium,GBCEE0300055,1992-05-26
1150690,We Will Rock You - Live,Queen Rock Montreal,GBCEE0700032,2007-10-09
1153059,We Will Rock You - Megan Thee Stallion Version,We Will Rock You (Megan Thee Stallion Version),GBUM72403911,2024-09-05
1530324,We Will Rock You - Movie Mix,Bohemian Rhapsody (The Original Soundtrack),GBUM71805976,2018-10-19
1581515,We Will Rock You,I Love Dad,GBCEE0100127,2006-06-05


In [153]:
import re

pattern = r"(\s-\s|\s\(|\s\[).*"

_df["track_name_clean"] = _df["track_name"].str.strip().str.lower().str.replace(pattern, "", regex=True).str.replace(r'\W+', '', regex=True)
_df = _df.drop_duplicates(subset=["artist_rowid", "track_name_clean"], keep="last").reset_index(drop=True)
print(len(_df))

1863421
1698452
1666340
1552107


## Data types

In [11]:
df

,track_rowid,track_name,artist_name,artist_rowid,album_rowid,album_name,album_type,label,release_date,release_date_precision,...,explicit,track_popularity,artist_popularity,artist_followers,album_popularity,duration_ms,track_number,disc_number,total_tracks,artist_genres
0,5,I Just Missed A Call,NOTD,1853555,2,Digital Notes,single,Universal Music AB,2025-03-14,day,...,0,51,62,317626,42,146424,4,1,6,<GENRE_UNKNOWN>
1,28,IT girl,JADE,5196797,15,FUFN (Fuck You For Now),single,RCA Records Label,2025-03-14,day,...,1,51,64,226527,51,153718,3,1,5,<GENRE_UNKNOWN>
2,129,Betrayal,Warren Zeiders,219151,37,"Relapse, Lies, & Betrayal",album,Warner Records,2025-03-14,day,...,0,51,75,934909,53,180282,3,1,21,country
3,138,Withdrawal,Warren Zeiders,219151,37,"Relapse, Lies, & Betrayal",album,Warner Records,2025-03-14,day,...,0,51,75,934909,53,185680,12,1,21,country
4,204,Afterall,Saveus,4490251,62,Afterall,single,The Bank Records,2025-03-14,day,...,0,51,47,40479,30,202688,1,1,1,dansktop
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
186955,2779,Not Like Us,Kendrick Lamar,4765104,358,Not Like Us,single,"Kendrick Lamar, under exclusive license to Int...",2024-05-04,day,...,1,96,97,39723389,85,274192,1,1,1,hip hop | west coast hip hop
186956,2985,BAILE INoLVIDABLE,Bad Bunny,2713710,373,DeBÍ TiRAR MáS FOToS,album,Rimas Entertainment LLC.,2025-01-05,day,...,1,96,100,93195263,100,367725,3,1,17,latin | reggaeton | trap latino | urbano latino
186957,269,BIRDS OF A FEATHER,Billie Eilish,842869,102,HIT ME HARD AND SOFT,album,Darkroom/Interscope Records,2024-05-17,day,...,0,98,95,109602603,94,210373,4,1,10,<GENRE_UNKNOWN>
186958,2998,DtMF,Bad Bunny,2713710,373,DeBÍ TiRAR MáS FOToS,album,Rimas Entertainment LLC.,2025-01-05,day,...,1,98,100,93195263,100,237117,16,1,17,latin | reggaeton | trap latino | urbano latino


In [20]:
df["mode"].unique()

array([1, 0])

In [17]:
for c in df.select_dtypes(include='number').columns:
    print(f"{c} {" " * (20 - len(c))} {df[c].dtype} \t {df[c].min():.2f} \t {df[c].max():.2f}")

track_rowid           int64 	 1.00 	 255162133.00
artist_rowid          int64 	 95.00 	 14296919.00
album_rowid           int64 	 1.00 	 58358798.00
time_signature        int64 	 0.00 	 5.00
tempo                 float64 	 0.00 	 243.37
key                   int64 	 0.00 	 11.00
mode                  int64 	 0.00 	 1.00
danceability          float64 	 0.00 	 0.99
energy                float64 	 0.00 	 1.00
loudness              float64 	 -54.38 	 4.92
speechiness           float64 	 0.00 	 0.97
acousticness          float64 	 0.00 	 1.00
instrumentalness      float64 	 0.00 	 1.00
liveness              float64 	 0.00 	 1.00
valence               float64 	 0.00 	 1.00
explicit              int64 	 0.00 	 1.00
track_popularity      int64 	 51.00 	 100.00
artist_popularity     int64 	 0.00 	 100.00
artist_followers      int64 	 0.00 	 141174367.00
album_popularity      int64 	 0.00 	 100.00
duration_ms           int64 	 30001.00 	 4500036.00
track_number          int64 	 1.00 	 198.00
dis

Set data types.

In [21]:
df = df.astype({
    # Identifiers
    'track_rowid': 'int64',
    'artist_rowid': 'int64',
    'album_rowid': 'int64',
    
    # Strings
    'track_name': 'string',
    'artist_name': 'string',
    'album_name': 'string',
    'label': 'string',
    'release_date': 'string',
    'id_isrc': 'string',
    'artist_genres': 'string',
    
    # Categoricals
    'album_type': 'category',
    'release_date_precision': 'category',
    
    # Audio features
    'tempo': 'float32',
    'danceability': 'float32',
    'energy': 'float32',
    'loudness': 'float32',
    'speechiness': 'float32',
    'acousticness': 'float32',
    'instrumentalness': 'float32',
    'liveness': 'float32',
    'valence': 'float32',
    
    # Small integers
    'time_signature': 'uint8',
    'key': 'uint8',
    'mode': 'uint8',
    'explicit': 'bool',
    'track_popularity': 'uint8',
    'artist_popularity': 'uint8',
    'album_popularity': 'uint8',
    'track_number': 'int16',
    'disc_number': 'uint8',
    'total_tracks': 'int16',
    
    # Larger integers
    'artist_followers': 'int64',
    'duration_ms': 'int32',
})

In [22]:
for c in df.select_dtypes(include='number').columns:
    print(f"{c} {" " * (20 - len(c))} {df[c].dtype} \t {df[c].min():.2f} \t {df[c].max():.2f}")

track_rowid           int64 	 1.00 	 255162133.00
artist_rowid          int64 	 95.00 	 14296919.00
album_rowid           int64 	 1.00 	 58358798.00
time_signature        uint8 	 0.00 	 5.00
tempo                 float32 	 0.00 	 243.37
key                   uint8 	 0.00 	 11.00
mode                  uint8 	 0.00 	 1.00
danceability          float32 	 0.00 	 0.99
energy                float32 	 0.00 	 1.00
loudness              float32 	 -54.38 	 4.92
speechiness           float32 	 0.00 	 0.97
acousticness          float32 	 0.00 	 1.00
instrumentalness      float32 	 0.00 	 1.00
liveness              float32 	 0.00 	 1.00
valence               float32 	 0.00 	 1.00
track_popularity      uint8 	 51.00 	 100.00
artist_popularity     uint8 	 0.00 	 100.00
artist_followers      int64 	 0.00 	 141174367.00
album_popularity      uint8 	 0.00 	 100.00
duration_ms           int32 	 30001.00 	 4500036.00
track_number          int16 	 1.00 	 198.00
disc_number           uint8 	 1.00 	 18.00
to